# LCoE sensitivity analysis — tornado chart


One-at-a-time sensitivity on each LCoE input. For each input, toggle ±20% from the base case and re-compute LCoE.

## Input
- `04_models/lcoe_workbook/lcoe_hybrid_system_v01.xlsx`

## Output
- `05_visualizations/charts/lcoe_sensitivity_tornado_v01.png`


In [ ]:
import sys
sys.path.insert(0, '../../../03_analysis/scripts')
import matplotlib.pyplot as plt
import numpy as np
from lcoe_model import lcoe, BASE_INPUTS, tornado_deltas   # shared single-source-of-truth model

base_lcoe = lcoe(**BASE_INPUTS)
print(f"Base LCoE: €{base_lcoe:.1f}/MWh")

## Computed sensitivity values

Each delta is computed by calling the shared LCoE model (`lcoe_model.lcoe`) at the base case and at each input's ±20% (or stated low/high) value — one input at a time. This is the same function the Monte Carlo uses, so the tornado and the distribution are guaranteed consistent. No values are hand-entered.


In [ ]:
from plotting_utils import tornado_chart

# Deltas computed from the shared model — one input at a time.
deltas = tornado_deltas()

fig, ax = tornado_chart(deltas, base_value=f'€{base_lcoe:.0f}/MWh',
                        title='LCoE sensitivity tornado (±20% one-at-a-time)')
items_sorted = sorted(deltas, key=lambda x: max(abs(x[1]), abs(x[2])))
for i, (label, low, high) in enumerate(items_sorted):
    ax.text(low,  i, f'{low:+.1f}',  va='center', ha=('right' if low  < 0 else 'left'),  fontsize=8)
    ax.text(high, i, f'{high:+.1f}', va='center', ha=('left'  if high > 0 else 'right'), fontsize=8)
plt.savefig('../../../05_visualizations/charts/lcoe_sensitivity_tornado_v01.png', dpi=150, bbox_inches='tight')
plt.savefig('../../../05_visualizations/charts/lcoe_sensitivity_tornado_v01.svg', bbox_inches='tight')
plt.show()

print(f"Base LCoE = €{base_lcoe:.1f}/MWh")
for label, low, high in sorted(deltas, key=lambda x: max(abs(x[1]), abs(x[2])), reverse=True):
    print(f"{label:30} low {low:+6.1f}   high {high:+6.1f}")

## Interpretation

Ranked by the computed LCoE swing (largest first):

1. **Heating FLEH (utilisation) — dominant (≈ +20 / −14 €/MWh).** LCoE is fixed-cost heavy, so the number of full-load-equivalent hours over which those costs are spread is the single biggest lever. A 20% drop in heating utilisation pushes LCoE well above the base.
2. **Discount rate (≈ ±8 €/MWh).** Financing terms drive the capital-recovery factor; subsidised vs commercial WACC is the main financing-side risk.
3. **Project lifetime (≈ +6 / −4 €/MWh)** and **electricity price (≈ ±6 €/MWh)** follow.
4. **Drilling cost (≈ ±4 €/MWh).**
5. **HP-1 COP (≈ +2 / −1 €/MWh)** and **USP-01 doublet flow (≈ +2 / −0.5 €/MWh)** are the *smallest* levers within a ±20% band.

**Why the well-deliverability risk is small here (and why this supersedes the earlier hand-entered tornado).** The system is demand-capped: it is designed to deliver a fixed 10 MWth heating peak regardless of the doublet flow. If the doublet under-delivers, the shortfall is made up efficiently by the heat pumps (extra electricity at COP 3.5–4.0) while doublet pumping electricity simultaneously falls — the two largely offset, so LCoE barely moves. The earlier version of this chart hand-entered "USP-01 = +€10/MWh (dominant)"; computing the sensitivity from the LCoE model shows that is not supported — **FLEH utilisation is the dominant driver.** The real USP-01 risk is a *discontinuous* one (inability to meet peak demand, or needing a second doublet), which a ±20% LCoE sensitivity does not capture and which is discussed in the risk section as a scenario, not a gradient.


In [ ]:
# --- CapEx breakdown (report §5.1 / LCoE workbook CapEx sheet) ---
capex = {'Subsurface\n(drilling, workover, permits)': 6.5,
         'Surface plant': 6.8,
         'ATES system': 2.0,
         'Pipelines': 5.3,
         'Contingency (15%)': 3.1}
cats, vals = list(capex), list(capex.values())
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(cats, vals, color=['#185FA5', '#0F6E56', '#E0A030', '#D85A30', '#9E9E9E'])
for b, v in zip(bars, vals):
    ax.text(v + 0.1, b.get_y() + b.get_height() / 2, f'\u20ac{v:.1f}M', va='center', fontsize=9, fontweight='bold')
ax.set_xlabel('CapEx (\u20acM)')
ax.set_title(f'Capital expenditure breakdown \u2014 total \u20ac{sum(vals):.1f}M', loc='left', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../../../05_visualizations/charts/capex_breakdown_v01.png', dpi=150, bbox_inches='tight')
plt.savefig('../../../05_visualizations/charts/capex_breakdown_v01.svg', bbox_inches='tight')
plt.show()